In [ ]:
import cv2
import pygame
import numpy as np
import socket
import threading
import json
import time


# =========================
# 서버 설정
# =========================

SERVER_IP = "10.10.59.205"
PORT = 5000
# 카메라 세팅
cap = cv2.VideoCapture(
    0
)

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)

MAX_CAMERA_RETRY = 5
CAMERA_RETRY_DELAY = 0.5

#앱 설정

WINDOW_NAME = "Python Game"

APP_WIDTH = 1280
APP_HEIGHT = 720

CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720

PANEL_X = 900
PANEL_WIDTH = 360



# =========================
# 네트워크 상태
# =========================

server_connected = False

player_id = -1
current_round = 0

scores = {}

round_winner = -1
game_winner = -1

already_sent = False


# =========================
# UI 알림 상태
# =========================

notice_title = ""
notice_subtitle = ""

notice_end_time = 0

NOTICE_DURATION = 2.0


# =========================
# 서버 메시지 전송
# =========================

def send_message(data):

    global server_connected

    if not server_connected:
        return

    try:

        message = json.dumps(data) + "\n"

        client_socket.sendall(
            message.encode("utf-8")
        )

    except Exception as e:

        print(
            "서버 전송 오류:",
            e
        )


# =========================
# 인식 성공 전송
# =========================

def send_detection():

    global already_sent

    if already_sent:
        return

    send_message({
        "type": "detected",
        "round": current_round
    })

    already_sent = True

    print(
        f"Round {current_round} 인식 성공 전송"
    )


# =========================
# 중앙 알림 표시 설정
# =========================

def show_notice(title, subtitle="", duration=2.0):

    global notice_title
    global notice_subtitle
    global notice_end_time

    notice_title = title
    notice_subtitle = subtitle

    notice_end_time = (
        time.time() + duration
    )


# =========================
# 서버 메시지 수신
# =========================

def receive_server():

    global player_id
    global current_round
    global scores

    global round_winner
    global game_winner

    global already_sent
    global server_connected
    global running

    global answer_index

    buffer = ""

    while server_connected:

        try:

            data = client_socket.recv(
                4096
            )

            if not data:

                print(
                    "서버 연결 종료"
                )

                server_connected = False

                break


            buffer += data.decode(
                "utf-8"
            )


            while "\n" in buffer:

                line, buffer = buffer.split(
                    "\n",
                    1
                )


                if not line.strip():
                    continue


                message = json.loads(
                    line
                )

                message_type = message.get(
                    "type"
                )


                # =========================
                # 서버 접속 완료
                # =========================

                if message_type == "connected":

                    player_id = message[
                        "player_id"
                    ]

                    current_round = message.get(
                        "round",
                        0
                    )

                    scores = message.get(
                        "scores",
                        {}
                    )

                    print(
                        f"Player {player_id}로 접속"
                    )


                # =========================
                # 라운드 시작
                # =========================

                elif message_type == "round_start":

                    current_round = message[
                        "round"
                    ]

                    scores = message.get(
                        "scores",
                        scores
                    )

                    answer_index = message.get(
                        "answer_index",
                        -1
                    )

                    round_winner = -1

                    already_sent = False

                    print(
                        f"Round {current_round} 시작"
                    )

                    print(
                        f"정답 Index : {answer_index}"
                    )

                    show_notice(
                        "라운드 시작!",
                        "물건을 찾아라!",
                        2.0
                    )


                # =========================
                # 라운드 종료
                # =========================

                elif message_type == "round_result":

                    round_winner = message[
                        "winner"
                    ]

                    scores = message[
                        "scores"
                    ]

                    print(
                        f"Round {message['round']} 종료"
                    )

                    print(
                        f"승자 : Player {round_winner}"
                    )

                    show_notice(
                        "라운드 끝!",
                        f"승자 : Player {round_winner}",
                        2.0
                    )


                # =========================
                # 게임 종료
                # =========================

                elif message_type == "game_over":

                    game_winner = message[
                        "winner"
                    ]

                    scores = message[
                        "scores"
                    ]

                    print(
                        f"게임 승자 : Player {game_winner}"
                    )

                    show_notice(
                        "게임 종료!",
                        f"최종 승자 : Player {game_winner}",
                        3.0
                    )

                    # 바로 종료하면 마지막 안내가 안 보이므로
                    # 여기서는 running=False 하지 않음


                # =========================
                # 플레이어 접속
                # =========================

                elif message_type == "player_joined":

                    scores = message.get(
                        "scores",
                        scores
                    )

                    print(
                        f"Player {message['player_id']} 접속"
                    )


                # =========================
                # 플레이어 연결 종료
                # =========================

                elif message_type == "player_left":

                    scores = message.get(
                        "scores",
                        scores
                    )

                    print(
                        f"Player {message['player_id']} 연결 종료"
                    )


        except Exception as e:

            if server_connected:

                print(
                    "서버 수신 오류:",
                    e
                )

            server_connected = False

            break


# =========================
# 좌측 상단 게임 정보 표시
# =========================

def display_game_info(
    surface,
    font,
    color
):

    x = 20
    y = 20

    line_height = 35


    # 현재 라운드
    round_surface = font.render(
        f"Round : {current_round}",
        True,
        color
    )

    surface.blit(
        round_surface,
        (x, y)
    )

    y += line_height + 10


    # Player ID 순서대로 표시
    for pid in sorted(
        scores,
        key=lambda value: int(value)
    ):

        score = scores[pid]

        # JSON으로 받은 dict key는 문자열일 수 있음
        pid_int = int(pid)

        if pid_int == player_id:

            text = (
                f"Player{pid_int}(나) : {score}"
            )

        else:

            text = (
                f"Player{pid_int} : {score}"
            )


        text_surface = font.render(
            text,
            True,
            color
        )

        surface.blit(
            text_surface,
            (x, y)
        )

        y += line_height


# =========================
# 중앙 알림 표시
# =========================

def display_notice(surface):

    if time.time() >= notice_end_time:
        return

    if notice_title == "":
        return


    # 반투명 검은 배경
    overlay = pygame.Surface(
        (
            surface.get_width(),
            180
        ),
        pygame.SRCALPHA
    )

    overlay.fill(
        (0, 0, 0, 160)
    )


    overlay_y = (
        surface.get_height() // 2
        - 90
    )


    surface.blit(
        overlay,
        (
            0,
            overlay_y
        )
    )


    # 제목
    title_surface = default_Font.render(
        notice_title,
        True,
        (255, 255, 255)
    )

    title_rect = title_surface.get_rect(
        center=(
            surface.get_width() // 2,
            surface.get_height() // 2 - 30
        )
    )

    surface.blit(
        title_surface,
        title_rect
    )


    # 부제목
    if notice_subtitle:

        subtitle_surface = default_Font.render(
            notice_subtitle,
            True,
            (255, 255, 255)
        )

        subtitle_rect = subtitle_surface.get_rect(
            center=(
                surface.get_width() // 2,
                surface.get_height() // 2 + 30
            )
        )

        surface.blit(
            subtitle_surface,
            subtitle_rect
        )


# =========================
# 게임 시작
# =========================

print(
    "게임 시작"
)

screen = pygame.display.set_mode(
    (
        APP_WIDTH,
        APP_HEIGHT
    )
)

pygame.display.set_caption(
    WINDOW_NAME
)


running = True
answer_index = -1


# =========================
# 카메라 설정
# =========================

cap = cv2.VideoCapture(
    0
)

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


# =========================
# 이미지 로드
# =========================

DIR = "../dataset"

images = [
    pygame.image.load(
        f"{DIR}/scraper/scraper_0000.jpg"
    ).convert_alpha(),

    pygame.image.load(
        f"{DIR}/scraper/scraper_0001.jpg"
    ).convert_alpha(),

    pygame.image.load(
        f"{DIR}/scraper/scraper_0002.jpg"
    ).convert_alpha(),

    pygame.image.load(
        f"{DIR}/nipper/nipper_0000.jpg"
    ).convert_alpha(),

    pygame.image.load(
        f"{DIR}/nipper/nipper_0001.jpg"
    ).convert_alpha()
]


time.sleep(
    1.0
)


# =========================
# 서버 연결
# =========================

client_socket = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)


try:

    client_socket.connect(
        (
            SERVER_IP,
            PORT
        )
    )

    server_connected = True

    print(
        "서버 연결 성공"
    )


except Exception as e:

    print(
        "서버 연결 실패:",
        e
    )


# =========================
# 서버 수신 스레드
# =========================

if server_connected:

    receive_thread = threading.Thread(
        target=receive_server,
        daemon=True
    )

    receive_thread.start()


# =========================
# 메인 루프
# =========================

while running:

    # -------------------------
    # 키 입력
    # -------------------------

    key_1_pressed = False


    for event in pygame.event.get():

        if event.type == pygame.QUIT:

            running = False


        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_e:

                running = False


            if event.key == pygame.K_1:

                key_1_pressed = True


    # -------------------------
    # 카메라
    # -------------------------

    successCapture, org_img = (
        cap.read()
    )


    if (
        not successCapture
        or org_img is None
    ):

        tryIndex += 1

        print(
            f"카메라 읽기 실패 "
            f"({tryIndex}/{MAX_CAMERA_RETRY})"
        )

        time.sleep(
            CAMERA_RETRY_DELAY
        )


        if (
            tryIndex
            >= MAX_CAMERA_RETRY
        ):

            print(
                "카메라 재연결 시도"
            )

            cap.release()

            time.sleep(
                0.5
            )

            cap = cv2.VideoCapture(
                0
            )

            cap.set(
                cv2.CAP_PROP_FRAME_WIDTH,
                1280
            )

            cap.set(
                cv2.CAP_PROP_FRAME_HEIGHT,
                720
            )

            tryIndex = 0

        continue


    tryIndex = 0


    # =========================
    # 카메라 그래픽 처리
    # =========================

    org_img = cv2.flip(
        org_img,
        1
    )


    org_img = cv2.cvtColor(
        org_img,
        cv2.COLOR_BGR2RGB
    )


    camera_surface = (
        pygame.surfarray.make_surface(
            np.transpose(
                org_img,
                (
                    1,
                    0,
                    2
                )
            )
        )
    )


    camera_surface = pygame.transform.scale(
        camera_surface,
        (
            CAMERA_WIDTH,
            CAMERA_HEIGHT
        )
    )


    # =========================
    # 화면 초기화
    # =========================

    screen.fill(
        (
            30,
            30,
            30
        )
    )


    screen.blit(
        camera_surface,
        (
            0,
            0
        )
    )


    # =========================
    # 정답 이미지
    # =========================

    if (
        answer_index >= 0
        and answer_index < len(images)
    ):

        display_image(
            screen,
            images[answer_index],
            APP_WIDTH - 300,
            APP_HEIGHT - 200,
            0.5
        )


    # =========================
    # 좌측 상단
    # 라운드 + 점수
    # =========================

    display_game_info(
        screen,
        default_Font,
        default_Color
    )


    # =========================
    # 테두리
    # =========================

    draw_screen_border(
        screen,
        default_Color,
        10
    )


    # =========================
    # 기본 안내
    # =========================

    display_text_centered(
        "물건을 찾아라",
        default_Font,
        default_Color,
        screen
    )


    # =========================
    # 라운드 시작/종료 알림
    # =========================

    display_notice(
        screen
    )


    # =========================
    # 테스트 입력
    # =========================

    if (
        key_1_pressed
        and not already_sent
    ):

        send_detection()


    # =========================
    # 실제 이미지 인식
    # =========================

    # detected = 이미지인식함수(org_img)
    #
    # if detected and not already_sent:
    #     send_detection()


    # =========================
    # 게임 종료 처리
    # =========================

    if (
        game_winner != -1
        and time.time() >= notice_end_time
    ):

        running = False


    # =========================
    # 화면 출력
    # =========================

    pygame.display.flip()


# =========================
# 종료
# =========================

cap.release()


try:

    client_socket.close()

except:

    pass


pygame.quit()

print(
    "프로그램 종료"
)

[ WARN:0@29.010] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x2de70600] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@29.010] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
[ WARN:0@29.182] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x2de70600] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@29.182] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


게임 시작
서버 연결 성공
Player 4로 접속
Player 4 접속


NameError: name 'tryIndex' is not defined

Player 3 연결 종료


: 